In [1]:
import os
import torch
import pandas as pd

from tqdm import tqdm

from stock_gpt import StockGPT, LinearModel, NaiveModel
from dataloader_builder import build_dataloaders
from setup import StockGPT_cfg, LinearModel_cfg, NaiveModel_cfg
from setup import path_data_preprocessor, PATH_RESULTS_NON_RESIDUALS, PATH_RESULTS_RESIDUALS
from model_training import model_setup, train_model_cuda

from model_training import train_model_cuda, evaluate_model, evaluate_best_model
from model_analysis import test_model, print_loss_analysis, process_losses, format_num, process_result, store_result

In [2]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## MODEL TRAINING ---------------------------

In [3]:
torch.manual_seed(1234)
dls, train_norms = build_dataloaders(path_data_preprocessor)

Building DataLoaders...


In [4]:
optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

max_epochs = 25

eval_bs = 1000

stockGPT, stockGPT_params, opt1, sca1, sch1 = model_setup(StockGPT, StockGPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, opt2, sca2, sch2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)

Input Norm: torch.Size([12])|torch.Size([12])
Target Norm: torch.Size([4])|torch.Size([4])
3181568
5376


NaiveModel()

In [5]:
model_train_losses, model_val_losses = train_model_cuda(stockGPT, device, opt1, sca1, sch1, max_epochs, 
                                                        dls["train"], dls["val"], eval_bs)
linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, opt2, sca2, sch2, max_epochs,
                                                        dls["train"], dls["val"], eval_bs)


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 1:

Learning Rate: 4.00e-04



|▍         | 4.0% (06:15) Evaluating model on validation data... (524/525) [5155/128875]:                 

Epoch 1:
Training Loss:
   (MAE) 0.0012648681877180934
   (NLL) -3.3748669624328613
Validation Loss:
   (MAE) 0.0011271049734205008
   (NLL) -3.3743934631347656

Best Validation: -3.3743934631347656
----------------------------------------------------------------------------------------------------

Learning Rate: 4.00e-04



|▊         | 8.0% (12:26) Evaluating model on validation data... (524/525) [10310/128875]: 

...

|█████▏    | 52.0% (1:23:34) Evaluating model on validation data... (524/525) [67015/128875]: 

Epoch 13:
Training Loss:
   (MAE) 0.0009703419636934996
   (NLL) -4.781675338745117
Validation Loss:
   (MAE) 0.0008737897151149809
   (NLL) -4.781213760375977

Best Validation: -4.781213760375977
----------------------------------------------------------------------------------------------------

Learning Rate: 4.00e-04



|█████▌    | 56.0% (1:29:58) Evaluating model on validation data... (524/525) [72170/128875]: 

Epoch 14:
Training Loss:
   (MAE) 0.0008895032806321979
   (NLL) -4.880455017089844
Validation Loss:
   (MAE) 0.0007913507870398462
   (NLL) -4.889770984649658

Best Validation: -4.889770984649658
----------------------------------------------------------------------------------------------------

Learning Rate: 4.00e-04



KeyboardInterrupt: 

## Model Analysis -------------------------

In [6]:
#* REUSES OBJETCS FROM TRAINING
analysis_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"])) + min(eval_bs, len(dls["test"]))
analysis_pbar = tqdm(total=3*analysis_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Reevaluates models by their best parameters on train and val dataloaders
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_bs, analysis_pbar)
linear_losses = evaluate_best_model(linearModel, device, opt2, sca2, sch2, dls["train"], dls["val"], eval_bs, analysis_pbar, True)
gpt_losses = evaluate_best_model(stockGPT, device, opt1, sca1, sch1, dls["train"], dls["val"], eval_bs, analysis_pbar, True) 

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, analysis_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, analysis_pbar)
gpt_test_losses = test_model(dls["test"], stockGPT, device, eval_bs, analysis_pbar)


|██████████| 100.0% (01:55) Evaluating model on testing data... (322/323) [5544/5544]:                    

|██████████| 100.0% (02:05) Evaluating model on testing data... (322/323) [5544/5544]: 

In [7]:
for key, features in [("NLL", StockGPT_cfg["target_features"]),
                      ("STD", [f"{feature}_std" for feature in StockGPT_cfg["target_features"]]),
                      ("MAE", StockGPT_cfg["target_features"]),
                      ("PMAE", StockGPT_cfg["target_features"])]:
    print_loss_analysis(process_losses(gpt_losses + gpt_test_losses +
                                       linear_losses + linear_test_losses +
                                       naive_losses + naive_test_losses, key), 
                                       [stockGPT.cfg["name"], linearModel.cfg["name"], naiveModel.cfg["name"]],
                                       [format_num(stockGPT_params), format_num(linearModel_params), "0"], 
                                       features, key)


--------------------------------------------------------------------------------------------------------------

NLL

--------------------------------------------------------------------------------------------------------------

                    o        h        l        c        
StockGPT-B5: 3.2M
    Training:       -4.9118  -4.8571  -4.8968  -4.8561    >  -4.8805
    Validation:     -4.9244  -4.8674  -4.9004  -4.8669    >  -4.8898
    Testing:        -4.9539  -4.8927  -4.9299  -4.8965    >  -4.9182
    
LinearModel-B5: 5.4K
    Training:       -3.9434  -3.8291  -3.8591  -3.7948    >  -3.8566
    Validation:     -3.8328  -3.7294  -3.7568  -3.7232    >  -3.7605
    Testing:        -3.8879  -3.7751  -3.8162  -3.7838    >  -3.8158
    
NaiveModel-B5: 0
    Training:       1.0953   1.0955   1.0951   1.0953     >  1.0953
    Validation:     1.1359   1.1360   1.1358   1.1359     >  1.1359
    Testing:        1.1367   1.1368   1.1366   1.1367     >  1.1367
    

-----------------------

In [8]:
import importlib
import setup
importlib.reload(setup)
from setup import PATH_RESULTS_RESIDUALS

store_result(PATH_RESULTS_RESIDUALS, process_result(stockGPT, gpt_losses, gpt_test_losses, max_epochs))
store_result(PATH_RESULTS_RESIDUALS, process_result(linearModel, linear_losses, linear_test_losses, max_epochs))
store_result(PATH_RESULTS_RESIDUALS, process_result(naiveModel, naive_losses, naive_test_losses, max_epochs))

print(pd.read_parquet(PATH_RESULTS_RESIDUALS))

{'model': 'StockGPT-B5', 'bar_width': 5, 'train': {'NLL': [-4.91178560256958, -4.857054710388184, -4.8968400955200195, -4.856138229370117], 'STD': [0.0028161099180579185, 0.002986591774970293, 0.0028423478361219168, 0.002949137706309557], 'MAE': [0.0009217888000421226, 0.0008762735524214804, 0.0008596159750595689, 0.0009003349114209414], 'PMAE': [34210616.0, 32840882.0, 32445970.0, 38492620.0]}, 'val': {'NLL': [-4.924426078796387, -4.867368221282959, -4.900424480438232, -4.866865634918213], 'STD': [0.0028034388087689877, 0.0029740140307694674, 0.002828237833455205, 0.002935758326202631], 'MAE': [0.00081919931108132, 0.0007749539217911661, 0.0007760325679555535, 0.0007952173473313451], 'PMAE': [51213908.0, 44866736.0, 47028164.0, 56400684.0]}, 'test': {'NLL': [-4.953893184661865, -4.892675399780273, -4.929927825927734, -4.896487236022949], 'STD': [0.0028251702897250652, 0.0029946209397166967, 0.0028489124961197376, 0.0029553768690675497], 'MAE': [0.0007454812293872237, 0.000707287399563

In [ ]:
from data_scrapper import scrape_data, get_all_tickers
from data_filler import fill_data
from data_preprocessor import preprocess_data
from setup import API_KEY, TIMEFRAME
import pandas_market_calendars as mcal

In [ ]:
return ""
all_tickers = get_all_tickers("raw_data/all_tickers_trimmed_1_30", API_KEY)
scrape_data(API_KEY,
              "raw_data/data_5min_2026",
              250,
              all_tickers,
              mcal.get_calendar("NYSE").schedule("2026-01-01","2026-7-1").index,
              TIMEFRAME)
fill_data("raw_data/data_5min_2026",
          "filled_raw_data/data_5min_2026",
          mcal.get_calendar("NYSE").schedule("2026-01-01","2026-7-1").index)
preprocess_data("filled_raw_data/data_5min_2026",
                "preprocessed_data/data_5min_2026",
                mcal.get_calendar("NYSE").schedule("2026-01-01","2026-7-1").index, [0.75, 0.9])

SyntaxError: 'return' outside function (521317128.py, line 1)

In [9]:
dls, train_norms = build_dataloaders("preprocessed_data/data_1min_2026")

Building DataLoaders...


In [10]:
test_losses = test_model(dls["test"], stockGPT, device, eval_bs)
print(test_losses)

({'NLL': tensor([-4.9357, -4.8962, -4.9225, -4.8984], device='cuda:0'), 'STD': tensor([0.0026, 0.0028, 0.0026, 0.0028], device='cuda:0'), 'MAE': tensor([0.0011, 0.0011, 0.0011, 0.0011], device='cuda:0'), 'PMAE': tensor([1.6053e+08, 1.5961e+08, 1.6146e+08, 1.6805e+08], device='cuda:0')},)
